In [0]:

-- ===========================================================
-- SECTION 4 : SUBQUERIES & CTEs
-- ============================================================
 
-- Q11. CTE: Customers who ordered more than once
WITH repeat_customers AS (
    SELECT
        customer_id,
        COUNT(order_id) AS order_count
    FROM orders
    WHERE order_status = 'Delivered'
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
)
SELECT
    c.customer_name,
    c.city,
    rc.order_count
FROM customers c
JOIN repeat_customers rc ON c.customer_id = rc.customer_id
ORDER BY rc.order_count DESC;
 
-- Q12. CTE: Average delivery time per agent and flag slow agents
WITH agent_stats AS (
    SELECT
        da.agent_name,
        da.city,
        AVG(od.pickup_time + od.drop_time) AS avg_delivery_time,
        COUNT(od.delivery_id)              AS total_deliveries
    FROM delivery_agents da
    JOIN order_delivery od ON da.agent_id = od.agent_id
    GROUP BY da.agent_id, da.agent_name, da.city
)
SELECT
    agent_name,
    city,
    ROUND(avg_delivery_time, 1) AS avg_delivery_mins,
    total_deliveries,
    CASE
        WHEN avg_delivery_time > 35 THEN 'Slow'
        WHEN avg_delivery_time > 28 THEN 'Average'
        ELSE 'Fast'
    END AS performance_flag
FROM agent_stats
ORDER BY avg_delivery_time;
 
-- Q13. Subquery: Restaurants earning above platform average revenue
SELECT
    r.restaurant_name,
    r.cuisine,
    SUM(o.order_amount) AS total_revenue
FROM restaurants r
JOIN orders o ON r.restaurant_id = o.restaurant_id
WHERE o.order_status = 'Delivered'
GROUP BY r.restaurant_id, r.restaurant_name, r.cuisine
HAVING SUM(o.order_amount) > (
    SELECT AVG(restaurant_revenue)
    FROM (
        SELECT SUM(order_amount) AS restaurant_revenue
        FROM orders
        WHERE order_status = 'Delivered'
        GROUP BY restaurant_id
    ) avg_table
)
ORDER BY total_revenue DESC;
 
-- Q14. Multi-CTE: Full customer order summary
WITH delivered AS (
    SELECT * FROM orders WHERE order_status = 'Delivered'
),
customer_summary AS (
    SELECT
        customer_id,
        COUNT(order_id)     AS total_orders,
        SUM(order_amount)   AS total_spend,
        AVG(order_amount)   AS avg_order,
        MAX(order_amount)   AS max_order,
        MIN(order_amount)   AS min_order
    FROM delivered
    GROUP BY customer_id
),
customer_segments AS (
    SELECT
        customer_id,
        total_orders,
        total_spend,
        CASE
            WHEN total_spend >= 1500 THEN 'High Value'
            WHEN total_spend >= 800  THEN 'Mid Value'
            ELSE 'Low Value'
        END AS segment
    FROM customer_summary
)
SELECT
    c.customer_name,
    c.city,
    cs.total_orders,
    cs.total_spend,
    cs.segment
FROM customers c
JOIN customer_segments cs ON c.customer_id = cs.customer_id
ORDER BY cs.total_spend DESC;
 
 
